In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
%matplotlib inline


## Завантаження датасету

In [2]:
df = pd.read_csv("../data/exam_dataset_1.csv")
df.head()


Розмір датасету: (1000, 8)
Колонки: ['id', 'category', 'X1', 'X2', 'X3', 'X4', 'X5', 'target']


,id,category,X1,X2,X3,X4,X5,target
0,0,4,7.719696,7.225326,6.591903,9.833928,7.503402,69.041729
1,1,4,5.663814,6.655295,10.396427,11.092169,8.209244,79.548400
2,2,3,7.910570,8.705756,9.554958,8.416181,6.257779,-71.008700
3,3,4,8.480861,6.883886,8.928048,9.050284,5.415433,-11.776024
4,4,5,5.965807,7.127756,8.126778,10.265996,7.239032,0.272058


## Базова інформація та статистика

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   id        1000 non-null   int64  
 1   category  1000 non-null   int64  
 2   X1        1000 non-null   float64
 3   X2        1000 non-null   float64
 4   X3        1000 non-null   float64
 5   X4        1000 non-null   float64
 6   X5        1000 non-null   float64
 7   target    1000 non-null   float64
dtypes: float64(6), int64(2)
memory usage: 62.6 KB


In [ ]:
df.describe().round(4)

,id,category,X1,X2,X3,X4,X5,target
count,1000.0000,1000.0000,1000.0000,1000.0000,1000.0000,1000.0000,1000.0000,1000.0000
mean,499.5000,3.0570,7.0356,7.9811,9.0455,10.0604,5.9975,16.4344
std,288.8194,1.4275,0.9998,0.9818,0.9735,0.9739,1.0005,108.6227
min,0.0000,1.0000,3.6931,4.4976,5.7959,6.7497,2.7574,-337.8288
25%,249.7500,2.0000,6.3528,7.3144,8.3511,9.4095,5.3462,-57.8348
50%,499.5000,3.0000,7.0606,7.9662,9.0943,10.0473,5.9971,15.2761
75%,749.2500,4.0000,7.7616,8.6751,9.7083,10.7142,6.6509,86.1562
max,999.0000,5.0000,9.7204,10.9317,12.1536,13.8099,9.3691,341.9767


## Визначення ознакової змінної з найбільшим середнім

Ознакові змінні — всі, **окрім** `target`, `category` та `id`.


In [5]:
feature_cols = [c for c in df.columns if c not in ["target", "category", "id"]]
print("Ознакові змінні:", feature_cols)

means = df[feature_cols].mean().sort_values(ascending=False)
print("\nСередні значення ознакових змінних:")
print(means.to_string())

target_col = means.idxmax()
print(f"\n→ Змінна з найбільшим середнім: [{target_col}] = {means.max():.4f}")

Ознакові змінні: ['X1', 'X2', 'X3', 'X4', 'X5']

Середні значення ознакових змінних:
X4    10.060438
X3     9.045542
X2     7.981056
X1     7.035569
X5     5.997488

→ Змінна з найбільшим середнім: [X4] = 10.0604


## Аналіз викидів у вибраній змінній (IQR-метод)

In [7]:
Q1  = df[target_col].quantile(0.25)
Q3  = df[target_col].quantile(0.75)
IQR = Q3 - Q1

lower_fence = Q1 - 1.5 * IQR
upper_fence = Q3 + 1.5 * IQR

outlier_mask = (df[target_col] < lower_fence) | (df[target_col] > upper_fence)
outliers     = df[outlier_mask]
print(f"\nКількість викидів: {len(outliers)}")
print("\nРядки з викидами:")
outliers

Змінна:       X4
Q1          = 9.4095
Q3          = 10.7142
IQR         = 1.3047
Нижня межа  = Q1 - 1.5·IQR = 7.4525
Верхня межа = Q3 + 1.5·IQR = 12.6713

Кількість викидів: 4

Рядки з викидами:


,id,category,X1,X2,X3,X4,X5,target
233,233,3,7.922828,7.848955,9.180330,13.110334,5.682337,332.464845
267,267,1,7.004770,7.100219,8.541865,6.749732,6.306584,-279.388479
413,413,3,6.503690,9.230840,9.485007,13.809884,6.509480,341.976747
898,898,3,7.788138,7.706913,7.709423,7.439710,5.013932,-201.823382


## Видалення викидів

In [9]:
df_clean = df[(df[target_col] >= lower_fence) & (df[target_col] <= upper_fence)].reset_index(drop=True)

print(f"Розмір до очищення:  {df.shape}")
print(f"Розмір після очищення: {df_clean.shape}")
print(f"Видалено рядків: {df.shape[0] - df_clean.shape[0]}")


Розмір до очищення:  (1000, 8)
Розмір після очищення: (996, 8)
Видалено рядків: 4


### Порівняння статистик до та після очищення

In [10]:
comparison = pd.DataFrame({
    "До очищення":    df[target_col].describe(),
    "Після очищення": df_clean[target_col].describe()
}).round(4)
comparison


,До очищення,Після очищення
count,1000.0000,996.0000
mean,10.0604,10.0596
std,0.9739,0.9544
min,6.7497,7.5081
25%,9.4095,9.4117
50%,10.0473,10.0473
75%,10.7142,10.7126
max,13.8099,12.6428


In [12]:
print(f"Фінальний датасет: {df_clean.shape[0]} рядків × {df_clean.shape[1]} колонок")
df_clean

Фінальний датасет: 996 рядків × 8 колонок


,id,category,X1,X2,X3,X4,X5,target
0,0,4,7.719696,7.225326,6.591903,9.833928,7.503402,69.041729
1,1,4,5.663814,6.655295,10.396427,11.092169,8.209244,79.548400
2,2,3,7.910570,8.705756,9.554958,8.416181,6.257779,-71.008700
3,3,4,8.480861,6.883886,8.928048,9.050284,5.415433,-11.776024
4,4,5,5.965807,7.127756,8.126778,10.265996,7.239032,0.272058
...,...,...,...,...,...,...,...,...
991,995,2,7.702462,8.337469,7.733702,11.268924,6.022390,165.471578
992,996,1,7.331130,8.474616,10.186114,10.348894,5.881496,58.913159
993,997,4,7.296364,9.262702,8.346836,9.890113,5.912725,20.335835
994,998,2,6.877261,7.435634,9.639704,9.904809,4.986812,-34.612815
